# 12 · Feature Engineering v4

Agrega ~70 columnas nuevas al `dataset_fe_v3.parquet` existente.  
Output: `dataset_fe_v4.parquet` (~390-400 columnas).

## Qué se agrega (no se superpone con v3)

| Grupo | Descripción | Columnas aprox |
|-------|-------------|----------------|
| 1 | Ratios de comportamiento (vaciamiento, deuda, canales) | ~15 |
| 2 | Lags/deltas de variables sin derivadas temporales en v3 | ~12 |
| 3 | Lags/deltas de los ratios nuevos del Grupo 1 | ~45 |

**Proyecto:** Predicción de Churn Bancario · Data Mining en EyF

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gc
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

RUTA_BASE = Path('/content/drive/MyDrive/Maestría Data Mining/DM en EyF')
RUTA_COMP = RUTA_BASE / 'Competencia 1'

CKPT_FE_V3 = RUTA_COMP / 'dataset_fe_v3.parquet'
CKPT_FE_V4 = RUTA_COMP / 'dataset_fe_v4.parquet'

COL_ID      = 'numero_de_cliente'
COL_PERIODO = 'foto_mes'

def mem_mb(df): return df.memory_usage(deep=True).sum() / 1024**2

print(f'FE v3 existe: {CKPT_FE_V4.exists() == False and CKPT_FE_V3.exists()}')
print(f'FE v4 ya existe: {CKPT_FE_V4.exists()} {"(se sobreescribirá)" if CKPT_FE_V4.exists() else ""}'  )

FE v3 existe: True
FE v4 ya existe: False 


## 1 · Carga del FE v3

In [3]:
print('Cargando dataset_fe_v3.parquet...')
df = pd.read_parquet(CKPT_FE_V3)
df = df.sort_values([COL_ID, COL_PERIODO]).reset_index(drop=True)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]} — {mem_mb(df):.1f} MB')
cols_originales = set(df.columns)
print(f'Columnas originales: {len(cols_originales)}')

Cargando dataset_fe_v3.parquet...
Shape: 983,061 × 327 — 1041.6 MB
Columnas originales: 327


---
## 2 · Grupo 1 — Ratios de comportamiento

Capturas relaciones entre variables que los lags solos no pueden ver.
Cada ratio tiene una interpretación de negocio directa.

In [4]:
print('=== GRUPO 1: Ratios de comportamiento ===')
eps = 1e-6  # evitar división por cero

# ── Transferencias ────────────────────────────────────────────────────────────
# ¿El cliente está sacando más plata de la que entra?
df['ratio_transfer_neto'] = (
    df['mtransferencias_emitidas'] /
    (df['mtransferencias_recibidas'].abs() + eps)
)
# Flujo neto en pesos: positivo = recibe más de lo que emite
df['net_transfer_pesos'] = (
    df['mtransferencias_recibidas'] - df['mtransferencias_emitidas']
)

# ── Deuda total y presión financiera ─────────────────────────────────────────
df['mdeuda_total'] = (
    df['mprestamos_personales'].fillna(0) +
    df['mprestamos_prendarios'].fillna(0) +
    df['mprestamos_hipotecarios'].fillna(0)
)
# ¿Cuántos sueldos necesita para cubrir su deuda total?
df['presion_deuda_payroll'] = (
    df['mdeuda_total'] / (df['mpayroll'].abs() + eps)
)
# ¿Qué % de su deuda es personal (más fácil de cancelar y migrar)?
df['pct_deuda_personal'] = (
    df['mprestamos_personales'].fillna(0) / (df['mdeuda_total'] + eps)
)

# ── Ticket promedio por tarjeta ───────────────────────────────────────────────
# Consumo por transacción — caída señala menor uso
df['ticket_visa'] = (
    df['mtarjeta_visa_consumo'] /
    (df['ctarjeta_visa_transacciones'] + eps)
)
df['ticket_master'] = (
    df['mtarjeta_master_consumo'] /
    (df['ctarjeta_master_transacciones'] + eps)
)
df['ticket_debito'] = (
    df['mautoservicio'] /
    (df['ctarjeta_debito_transacciones'] + eps)
)

# ── Presión de débitos automáticos sobre el saldo ────────────────────────────
df['ratio_debitos_saldo'] = (
    df['mcuenta_debitos_automaticos'].fillna(0) /
    (df['mcuentas_saldo'].abs() + eps)
)

# ── Inversiones totales y ratio sobre saldo ───────────────────────────────────
# ¿El cliente está retirando inversiones? (señal fuerte de churn)
df['monto_inversiones_total'] = (
    df['minversion1_pesos'].fillna(0) +
    df['minversion1_dolares'].fillna(0) +
    df['minversion2'].fillna(0) +
    df['mplazo_fijo_pesos'].fillna(0) +
    df['mplazo_fijo_dolares'].fillna(0)
)
df['ratio_inversion_saldo'] = (
    df['monto_inversiones_total'] /
    (df['mcuentas_saldo'].abs() + eps)
)

# ── Dolarización ──────────────────────────────────────────────────────────────
# ¿Está convirtiendo pesos a dólares? (puede preceder salida)
df['pct_saldo_dolares'] = (
    df['mcaja_ahorro_dolares'].fillna(0) /
    (df['mcuentas_saldo'].abs() + eps)
)

# ── Canal de interacción ──────────────────────────────────────────────────────
# Digital vs presencial — migración a presencial puede señalar fricción
df['ratio_canal_digital'] = (
    (df['chomebanking_transacciones'].fillna(0) +
     df['ctarjeta_debito_transacciones'].fillna(0)) /
    (df['ccajas_transacciones'].fillna(0) +
     df['ccallcenter_transacciones'].fillna(0) + eps)
)
# Actividad total de canales
df['actividad_canales_total'] = (
    df['chomebanking_transacciones'].fillna(0) +
    df['ctarjeta_debito_transacciones'].fillna(0) +
    df['ccajas_transacciones'].fillna(0) +
    df['ccallcenter_transacciones'].fillna(0)
)

cols_g1 = [c for c in df.columns if c not in cols_originales]
print(f'Columnas agregadas Grupo 1: {len(cols_g1)}')
for c in cols_g1:
    print(f'  {c}')

=== GRUPO 1: Ratios de comportamiento ===
Columnas agregadas Grupo 1: 14
  ratio_transfer_neto
  net_transfer_pesos
  mdeuda_total
  presion_deuda_payroll
  pct_deuda_personal
  ticket_visa
  ticket_master
  ticket_debito
  ratio_debitos_saldo
  monto_inversiones_total
  ratio_inversion_saldo
  pct_saldo_dolares
  ratio_canal_digital
  actividad_canales_total


---
## 3 · Grupo 2 — Lags y deltas de variables sin derivadas temporales en v3

`ccallcenter_transacciones` solo tenía las columnas base.
`ccajas_transacciones` tenía lag1 pero no delta.
Variables de débitos master sin derivadas temporales.

In [5]:
print('=== GRUPO 2: Lags/deltas de variables huérfanas ===')

# Variables que en v3 no tienen lag1 o delta1
VARS_HUERFANAS = [
    'ccallcenter_transacciones',           # solo base en v3
    'ctarjeta_master_debitos_automaticos', # sin derivadas
    'mttarjeta_master_debitos_automaticos',# sin derivadas
    'ccajas_depositos',                    # sin derivadas
]
# Solo las que existen en el dataset
VARS_HUERFANAS = [v for v in VARS_HUERFANAS if v in df.columns]

cols_antes = set(df.columns)

for var in VARS_HUERFANAS:
    lag_col = f'{var}_lag1'
    # Solo agregar lag1 si no existe ya
    if lag_col not in df.columns:
        df[lag_col] = df.groupby(COL_ID)[var].shift(1)

    delta_col = f'{var}_delta1'
    if delta_col not in df.columns:
        lag = df.groupby(COL_ID)[var].shift(1)
        df[delta_col] = df[var] - lag

    delta_rel_col = f'{var}_delta_rel1'
    if delta_rel_col not in df.columns:
        lag = df.groupby(COL_ID)[var].shift(1)
        df[delta_rel_col] = (df[var] - lag) / (lag.abs() + eps)

cols_g2 = [c for c in df.columns if c not in cols_originales and c not in cols_g1]
print(f'Columnas agregadas Grupo 2: {len(cols_g2)}')
for c in cols_g2:
    print(f'  {c}')
gc.collect()

=== GRUPO 2: Lags/deltas de variables huérfanas ===
Columnas agregadas Grupo 2: 12
  ccallcenter_transacciones_lag1
  ccallcenter_transacciones_delta1
  ccallcenter_transacciones_delta_rel1
  ctarjeta_master_debitos_automaticos_lag1
  ctarjeta_master_debitos_automaticos_delta1
  ctarjeta_master_debitos_automaticos_delta_rel1
  mttarjeta_master_debitos_automaticos_lag1
  mttarjeta_master_debitos_automaticos_delta1
  mttarjeta_master_debitos_automaticos_delta_rel1
  ccajas_depositos_lag1
  ccajas_depositos_delta1
  ccajas_depositos_delta_rel1


0

---
## 4 · Grupo 3 — Lags y deltas de los ratios del Grupo 1

Los ratios solos son una foto mensual.
El **cambio** del ratio es la señal de churn que el modelo necesita:
- `ratio_transfer_neto` subiendo → está sacando cada vez más
- `ratio_canal_digital` bajando → se aleja del banco digitalmente
- `ticket_visa` cayendo → está usando menos la tarjeta

In [6]:
print('=== GRUPO 3: Lags/deltas de ratios nuevos ===')

# Ratios del Grupo 1 sobre los que vale la pena calcular derivadas temporales
RATIOS_CON_DERIVADAS = [
    'ratio_transfer_neto',
    'net_transfer_pesos',
    'presion_deuda_payroll',
    'pct_deuda_personal',
    'ticket_visa',
    'ticket_master',
    'ticket_debito',
    'ratio_debitos_saldo',
    'ratio_inversion_saldo',
    'pct_saldo_dolares',
    'ratio_canal_digital',
    'actividad_canales_total',
    'mdeuda_total',
    'monto_inversiones_total',
]

cols_antes_g3 = set(df.columns)

for var in RATIOS_CON_DERIVADAS:
    if var not in df.columns:
        continue

    # Lag t-1
    df[f'{var}_lag1'] = df.groupby(COL_ID)[var].shift(1)

    # Delta absoluto t vs t-1
    lag1 = df.groupby(COL_ID)[var].shift(1)
    df[f'{var}_delta1'] = df[var] - lag1

    # Delta relativo
    df[f'{var}_delta_rel1'] = (df[var] - lag1) / (lag1.abs() + eps)

    # Media ventana 3 meses (señal de tendencia)
    df[f'{var}_ma3'] = (
        df.groupby(COL_ID)[var]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
    )

    gc.collect()

cols_g3 = [c for c in df.columns
           if c not in cols_originales and c not in cols_g1 and c not in cols_g2]
print(f'Columnas agregadas Grupo 3: {len(cols_g3)}')
for c in sorted(cols_g3):
    print(f'  {c}')

=== GRUPO 3: Lags/deltas de ratios nuevos ===
Columnas agregadas Grupo 3: 56
  actividad_canales_total_delta1
  actividad_canales_total_delta_rel1
  actividad_canales_total_lag1
  actividad_canales_total_ma3
  mdeuda_total_delta1
  mdeuda_total_delta_rel1
  mdeuda_total_lag1
  mdeuda_total_ma3
  monto_inversiones_total_delta1
  monto_inversiones_total_delta_rel1
  monto_inversiones_total_lag1
  monto_inversiones_total_ma3
  net_transfer_pesos_delta1
  net_transfer_pesos_delta_rel1
  net_transfer_pesos_lag1
  net_transfer_pesos_ma3
  pct_deuda_personal_delta1
  pct_deuda_personal_delta_rel1
  pct_deuda_personal_lag1
  pct_deuda_personal_ma3
  pct_saldo_dolares_delta1
  pct_saldo_dolares_delta_rel1
  pct_saldo_dolares_lag1
  pct_saldo_dolares_ma3
  presion_deuda_payroll_delta1
  presion_deuda_payroll_delta_rel1
  presion_deuda_payroll_lag1
  presion_deuda_payroll_ma3
  ratio_canal_digital_delta1
  ratio_canal_digital_delta_rel1
  ratio_canal_digital_lag1
  ratio_canal_digital_ma3
  ratio

---
## 5 · Limpieza y guardado

In [7]:
# Reemplazar infinitos generados por divisiones
print('Limpiando infinitos...')
cols_nuevas = [c for c in df.columns if c not in cols_originales]
for col in cols_nuevas:
    inf_mask = np.isinf(df[col])
    if inf_mask.any():
        df.loc[inf_mask, col] = np.nan

# Clip de outliers extremos en deltas relativos (>100x o <-100x)
delta_rel_cols = [c for c in cols_nuevas if 'delta_rel' in c]
for col in delta_rel_cols:
    df[col] = df[col].clip(-100, 100)

print(f'\nResumen final:')
print(f'  Columnas originales (v3): {len(cols_originales):,}')
print(f'  Grupo 1 (ratios):         {len(cols_g1):,}')
print(f'  Grupo 2 (huérfanas):      {len(cols_g2):,}')
print(f'  Grupo 3 (lags de ratios): {len(cols_g3):,}')
print(f'  TOTAL v4:                 {df.shape[1]:,}')
print(f'  Memoria:                  {mem_mb(df):.1f} MB')

# Verificar que no haya columnas duplicadas
assert df.columns.duplicated().sum() == 0, 'Hay columnas duplicadas!'
print('\n✓ Sin columnas duplicadas')

Limpiando infinitos...

Resumen final:
  Columnas originales (v3): 327
  Grupo 1 (ratios):         14
  Grupo 2 (huérfanas):      12
  Grupo 3 (lags de ratios): 56
  TOTAL v4:                 409
  Memoria:                  1504.7 MB

✓ Sin columnas duplicadas


In [8]:
print(f'Guardando dataset_fe_v4.parquet...')
t0 = datetime.now()
df.to_parquet(CKPT_FE_V4, index=False)
elapsed = (datetime.now() - t0).seconds
print(f'✓ Guardado en {elapsed}s — {CKPT_FE_V4.name}')
print(f'  Tamaño en disco: {CKPT_FE_V4.stat().st_size / 1024**2:.1f} MB')

del df; gc.collect()
print('\n✓ Notebook 12 completada. Input para la notebook 11: dataset_fe_v4.parquet')

Guardando dataset_fe_v4.parquet...
✓ Guardado en 15s — dataset_fe_v4.parquet
  Tamaño en disco: 599.2 MB

✓ Notebook 12 completada. Input para la notebook 11: dataset_fe_v4.parquet
